# DiceDial: Google Colab & Remote GPU Execution Notebook

This notebook automates the environment setup, dependency installation, simulation smoke testing, PPO training curriculum, multi-seed evaluation, and video rendering for **DiceDial** on Google Colab or an NVIDIA GPU instance.

## 1. Environment & Hardware Diagnostic
Verify that an NVIDIA GPU is available in your runtime environment.

In [ ]:
!nvidia-smi

## 2. Workspace Setup & Repository Cloning
Clone or sync the latest committed DiceDial repository and switch into the workspace directory.

In [ ]:
import os
if not os.path.exists('DiceDial'):
    !git clone https://github.com/dheerajdhillon/DiceDial.git
%cd DiceDial
!git pull

## 3. Dependency & Simulator Installation
Install PyTorch with CUDA support, NVIDIA Isaac Sim, Isaac Lab, and DiceDial in editable mode.

In [ ]:
# Install PyTorch with CUDA acceleration
!pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu128

# Install NVIDIA Isaac Sim
!pip install "isaacsim[all,extscache]" --extra-index-url https://pypi.nvidia.com

In [ ]:
# Install Isaac Lab from official repository
import os
if not os.path.exists('../IsaacLab'):
    %cd ..
    !git clone --depth 1 https://github.com/isaac-sim/IsaacLab.git
    %cd IsaacLab
    !./isaaclab.sh --install
    %cd ../DiceDial
else:
    print('IsaacLab directory already present.')

In [ ]:
# Install DiceDial package dependencies
!pip install -e ".[video,test]"

## 4. Verification & Smoke Testing
Run unit tests for geometry/assets followed by a full simulator environment smoke test.

In [ ]:
# Run unit tests for geometry and USD asset structure
!pytest

In [ ]:
# Run headless simulation smoke test
!python scripts/smoke_test.py --task DiceDial-Shadow-Random-v0 --num_envs 16 --steps 200 --headless

## 5. Model Training & Curriculum Execution
Run single stage debug training or execute the complete 3-stage curriculum (`Easy → Random → Sequence`).

In [ ]:
# Debug stage training (Single-command fixed goal)
!python scripts/train.py --task DiceDial-Shadow-Easy-v0 --num_envs 512 --total_timesteps 5000000 --run_name easy_debug --headless

In [ ]:
# Complete 3-Stage Curriculum Execution
!bash scripts/train_curriculum.sh

In [ ]:
# Plot task performance metrics
!python scripts/plot_metrics.py \
  --csv outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/task_metrics.csv \
  --output outputs/dicedial_training_metrics.png

## 6. Evaluation & Robustness Protocol
Run held-out seeds for nominal sequence tasks and randomized die mass/friction variations.

In [ ]:
# Run full 3-seed evaluation protocol
!scripts/run_final_evaluation.sh \
  outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model.zip \
  outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model_vecnormalize.pkl

## 7. Video Generation & Annotation
Render the continuous 6-face command sequence and produce the annotated MP4 video.

In [ ]:
# Render raw playback video
!python scripts/play.py \
  --task DiceDial-Shadow-Play-v0 \
  --model outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model.zip \
  --vecnormalize outputs/DiceDial-Shadow-Sequence-v0/stage3_sequence/model_vecnormalize.pkl \
  --output videos/final

In [ ]:
# Annotate video with live task telemetry overlay
!python scripts/annotate_video.py \
  --video videos/final/raw/dicedial-episode-0.mp4 \
  --metrics videos/final/video_metrics.csv \
  --output videos/final/dicedial_annotated.mp4